# 🤖 Módulo 3 · Cuaderno 2 — Supervisar a Nexo
### Trazabilidad, métricas, tablero de control, alertas y esquema de supervisión

**AVE Ingeniería de Prompting y Agentes Autónomos · Universidad Ean**

Viernes, 11 de la noche: el centro de acopio está cerrado, pero Nexo sigue respondiendo. ¿Cómo sabe la cooperativa si Nexo está funcionando bien? ¿Quién se entera si empieza a fallar? Un agente en producción necesita lo mismo que cualquier proceso importante: **registro, indicadores y alarmas**.

**Al terminar podrás:**
1. Registrar cada decisión del agente en una **traza** consultable.
2. Calcular **métricas de eficacia y seguridad** con su línea base.
3. Construir un **tablero de control** con gráficas.
4. Programar **alertas** que avisen a una persona a tiempo.
5. Diseñar un **plan de delegación progresiva** y tu **esquema de supervisión**.
6. Dejar listo tu cuaderno para la **entrega del proyecto final**.

> Relación con el AVE: Módulo 3, Págs. 06 a 10.

## 0. Antes de empezar: cómo usar este cuaderno

Un **cuaderno de Colab** mezcla texto (como este) y **celdas de código** que puedes ejecutar.

- Para ejecutar una celda, haz clic en ella y presiona **Shift + Enter** (o el botón ▶ de la izquierda).
- Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla, vuelve a ejecutar desde la sección de configuración.
- Lo que escribe el código aparece debajo de la celda: eso es la **salida**.
- Si cierras la pestaña o pasa mucho tiempo, Colab reinicia la sesión y hay que ejecutar todo otra vez (*Entorno de ejecución → Ejecutar todas*).

### Tu clave de Gemini (gratis)

Nexo usa los modelos **Gemini** de Google a través de su API. Necesitas una clave personal:

1. Entra a <https://aistudio.google.com/apikey> con tu cuenta de Google y haz clic en **Create API key**.
2. Copia la clave.
3. En Colab, abre el panel izquierdo y haz clic en el ícono de la **llave 🔑 (Secretos)**.
4. Agrega un secreto con nombre `GOOGLE_API_KEY`, pega la clave en *Valor* y activa **Acceso al cuaderno**.

> ⚠️ **Nunca escribas tu clave dentro del código** ni la compartas: si entregas o compartes el cuaderno, cualquiera podría usarla. Los *Secretos* de Colab la guardan aparte.

La capa gratuita de Gemini tiene un límite de solicitudes por minuto. Si ves un error **429**, espera un minuto: el código de este cuaderno ya reintenta solo.

## 1. Configuración

In [ ]:
# ── Instalación de librerías ─────────────────────────────────────────────
# El signo "!" ejecuta un comando de la terminal dentro de Colab.
# google-genai : la librería oficial de Google para usar los modelos Gemini.
# pydantic     : sirve para definir "moldes" de datos y pedir respuestas con estructura fija.
# -q (quiet)   : instala sin llenar la pantalla de mensajes.
!pip install -q -U google-genai pydantic

In [ ]:
# ── Configuración: conectar con Gemini ───────────────────────────────────
import time                      # para medir tiempos y esperar entre reintentos
import json                      # para leer y escribir datos en formato JSON
from google import genai         # cliente para hablar con los modelos Gemini
from google.genai import types   # "tipos": objetos de configuración (temperatura, herramientas, etc.)


def obtener_clave():
    """Lee la clave desde los Secretos de Colab; si no existe, la pide por teclado."""
    try:
        from google.colab import userdata
        clave = userdata.get("GOOGLE_API_KEY")
        if clave:
            return clave
    except Exception:
        pass  # no estamos en Colab o el secreto no tiene acceso al cuaderno
    from getpass import getpass  # getpass oculta lo que escribes
    return getpass("Pega tu clave de Gemini y presiona Enter: ")


# El "cliente" es el objeto que envía nuestras preguntas al modelo y trae las respuestas.
client = genai.Client(api_key=obtener_clave())

# Los nombres de los modelos cambian con el tiempo. Probamos una lista en orden de preferencia
# y nos quedamos con el primero que esté disponible para tu clave.
MODELOS_PREFERIDOS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-2.0-flash"]


def elegir_modelo(preferidos=MODELOS_PREFERIDOS):
    disponibles = []
    for m in client.models.list():
        acciones = m.supported_actions or []
        if "generateContent" in acciones:
            disponibles.append(m.name.replace("models/", ""))
    for nombre in preferidos:
        if nombre in disponibles:
            return nombre
    # Si ninguno de la lista existe, usamos cualquier modelo "flash" disponible.
    flash = [n for n in disponibles if "flash" in n and "image" not in n and "tts" not in n]
    return flash[0] if flash else disponibles[0]


MODELO = elegir_modelo()
print("✅ Conectado. Modelo que usará Nexo:", MODELO)


def llamar_con_reintentos(funcion, *args, intentos=5, **kwargs):
    """Ejecuta una llamada a la API y, si el servicio está saturado (error 429 o 503),
    espera y vuelve a intentar. Así el cuaderno no se detiene por límites de la capa gratuita."""
    espera = 10
    for intento in range(1, intentos + 1):
        try:
            return funcion(*args, **kwargs)
        except Exception as error:
            mensaje = str(error)
            if any(codigo in mensaje for codigo in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")) and intento < intentos:
                print(f"⏳ Servicio ocupado (intento {intento}). Espero {espera} s y reintento...")
                time.sleep(espera)
                espera *= 2  # cada vez esperamos el doble
            else:
                raise


def preguntar(prompt, sistema=None, temperatura=None, modelo=None, **otros):
    """Envía un texto al modelo y devuelve la respuesta como texto.
    - prompt:      lo que le pedimos al modelo.
    - sistema:     instrucciones de sistema (quién es y cómo se comporta Nexo).
    - temperatura: 0 = respuestas más estables; 1 o más = respuestas más variadas.
    - otros:       cualquier otro ajuste de GenerateContentConfig."""
    config = types.GenerateContentConfig(system_instruction=sistema, temperature=temperatura, **otros)
    respuesta = llamar_con_reintentos(
        client.models.generate_content, model=modelo or MODELO, contents=prompt, config=config
    )
    return respuesta.text


print(preguntar("Saluda en una sola frase, en español."))

In [ ]:
# ── Datos (ficticios) de la Cooperativa Raíces ───────────────────────────
# Son la "fuente de verdad" de la cooperativa. Nexo no debe inventar nada que no esté aquí.
COOPERATIVA = {
    "nombre": "Cooperativa Raíces",
    "producto": "café pergamino seco",
    "precio_referencia_carga": 2_150_000,   # pesos colombianos por carga de 125 kg (se actualiza cada lunes)
    "fecha_precio": "lunes de esta semana",
    "horario_acopio": "lunes a sábado, de 6:00 a. m. a 2:00 p. m.",
}

REGLAMENTO = [
    "Art. 1. Precio de referencia: la cooperativa publica cada lunes el precio de referencia por carga de 125 kg de café pergamino seco. Nexo solo informa el precio vigente publicado.",
    "Art. 2. Calidad: el café se recibe con humedad entre 10 % y 12 %. Si la humedad supera el 12 %, se descuenta el 1 % del valor por cada punto adicional.",
    "Art. 3. Horario del centro de acopio: lunes a sábado de 6:00 a. m. a 2:00 p. m. Los domingos y festivos no hay recepción.",
    "Art. 4. Recogida en camión: solo para entregas de 10 cargas o más, en veredas con vía carreteable, y se solicita con 48 horas de anticipación.",
    "Art. 5. Pagos: los pagos se hacen por transferencia a la cuenta registrada del asociado dentro de los 3 días hábiles siguientes a la entrega.",
    "Art. 6. Aprobación de pagos: todo pago igual o superior a 5.000.000 de pesos requiere la aprobación de la analista de pagos.",
    "Art. 7. Anticipos de cosecha: un asociado puede pedir un anticipo de hasta el 30 % del valor entregado el año anterior.",
    "Art. 8. Datos personales: Nexo no comparte datos de un asociado con otra persona y nunca pide contraseñas ni claves bancarias.",
    "Art. 9. Quejas: toda queja se registra con un número de caso y la responde una persona del equipo en máximo 5 días hábiles.",
    "Art. 10. Asamblea: la asamblea general de asociados se realiza el último sábado de marzo.",
]

ASOCIADOS = {
    "A-101": {"nombre": "Aurelio Gómez", "vereda": "La Esperanza", "via_carreteable": True,  "cargas_anio_anterior": 48},
    "A-102": {"nombre": "Rosa Pineda",   "vereda": "El Diviso",    "via_carreteable": False, "cargas_anio_anterior": 22},
    "A-103": {"nombre": "Julián Ortiz",  "vereda": "Alto Bonito",  "via_carreteable": True,  "cargas_anio_anterior": 9},
}

print("Precio de referencia:", f"${COOPERATIVA['precio_referencia_carga']:,}".replace(",", "."), "por carga")
print("Artículos del reglamento:", len(REGLAMENTO), "| Asociados registrados:", len(ASOCIADOS))

## 2. Trazabilidad: el registro de decisiones

Cada vez que Nexo actúa guardamos un **evento** con: cuándo, quién (qué agente), qué recibió, qué decidió, con qué herramienta, en qué nivel de autonomía, si una persona lo corrigió y cuánto tardó. Lo guardamos en un archivo **JSONL** (un JSON por línea), un formato muy usado para registros.

In [ ]:
import uuid
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field

ARCHIVO_TRAZA = "traza_nexo.jsonl"

@dataclass
class Evento:
    agente: str
    entrada: str
    decision: str
    herramienta: str = "ninguna"
    nivel: int = 1                       # 1 autónomo, 2 con aprobación, 3 escalado
    corregido_por_humano: bool = False
    error: bool = False
    latencia_s: float = 0.0
    momento: str = field(default_factory=lambda: datetime.now().isoformat(timespec="seconds"))
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

def registrar(evento: Evento):
    with open(ARCHIVO_TRAZA, "a", encoding="utf-8") as f:
        f.write(json.dumps(asdict(evento), ensure_ascii=False) + "\n")
    return evento.id

### Instrumentar a Nexo con un decorador
Un **decorador** (`@trazar`) envuelve una función para agregarle un comportamiento sin cambiar su código: aquí, medir el tiempo y registrar el evento automáticamente.

In [ ]:
import functools

def trazar(nombre_agente, herramienta="ninguna"):
    def envoltura(funcion):
        @functools.wraps(funcion)
        def interna(mensaje, *args, **kwargs):
            inicio = time.time()
            try:
                resultado, nivel = funcion(mensaje, *args, **kwargs)
                registrar(Evento(agente=nombre_agente, entrada=mensaje, decision=resultado[:300],
                                 herramienta=herramienta, nivel=nivel, latencia_s=round(time.time() - inicio, 2)))
                return resultado
            except Exception as e:
                registrar(Evento(agente=nombre_agente, entrada=mensaje, decision=str(e)[:300], error=True,
                                 latencia_s=round(time.time() - inicio, 2)))
                raise
        return interna
    return envoltura

SYSTEM_NEXO = ("Eres Nexo, asistente de la Cooperativa Raíces. Responde en máximo 3 frases usando solo el reglamento. "
               "Si es una queja o algo que no está en el reglamento, responde exactamente: ESCALAR.\n" + "\n".join(REGLAMENTO)
               + f"\nPrecio de referencia vigente: {COOPERATIVA['precio_referencia_carga']} pesos por carga.")

@trazar("Nexo")
def nexo(mensaje):
    respuesta = preguntar(mensaje, sistema=SYSTEM_NEXO, temperatura=0.2)
    nivel = 3 if "ESCALAR" in respuesta.upper() else 1
    return respuesta, nivel

for m in ["¿Abren el domingo?", "¿A cómo está la carga?", "El camión no llegó, estoy muy molesto"]:
    print(f"👤 {m}\n🤖 {nexo(m)}\n")

In [ ]:
import pandas as pd
traza = pd.read_json(ARCHIVO_TRAZA, lines=True)
traza[["momento", "agente", "entrada", "nivel", "latencia_s", "id"]]

## 3. Una semana de operación (datos simulados)

Para practicar con métricas y tableros necesitamos muchos eventos. En lugar de gastar cientos de llamadas a la API, **simulamos** dos semanas de operación con números aleatorios controlados (`seed`). En tu proyecto puedes usar tu traza real o una simulación, pero **decláralo**.

Simulamos que en la segunda semana la cooperativa cambió el precio y Nexo empezó a equivocarse más: veremos si el tablero lo detecta.

In [ ]:
import numpy as np
rng = np.random.default_rng(seed=7)    # seed: los mismos números aleatorios cada vez que se ejecuta

filas = []
inicio = datetime(2026, 11, 2, 6, 0)
for dia in range(14):
    for _ in range(rng.integers(25, 45)):              # entre 25 y 45 consultas por día
        # La mayoría de consultas llegan en horario de atención (6:00 a 14:00). El viernes de la
        # semana 1 (dia 4) simulamos un pico de mensajes nocturnos, como el de la Pág. 09 del AVE.
        prob_horario = 0.45 if dia == 4 else 0.85
        horas = rng.uniform(0, 8) if rng.random() < prob_horario else rng.uniform(8, 17.9)
        momento = inicio + timedelta(days=dia, hours=float(horas))
        semana2 = dia >= 7
        categoria = rng.choice(["precio", "recogida", "pago", "queja", "otro"], p=[.35, .25, .2, .1, .1])
        nivel = 3 if categoria == "queja" else (2 if categoria == "pago" and rng.random() < .4 else 1)
        prob_correccion = (.18 if semana2 and categoria == "precio" else .05)
        filas.append({
            "momento": momento, "dia": momento.date(), "semana": 2 if semana2 else 1,
            "categoria": categoria, "nivel": nivel,
            "corregido_por_humano": bool(rng.random() < prob_correccion),
            "error": bool(rng.random() < .02),
            "latencia_s": float(rng.gamma(2.0, 1.2)),
            "fuera_de_horario": not (6 <= momento.hour < 14),
        })
sim = pd.DataFrame(filas)
print("Eventos simulados:", len(sim))
sim.head()

## 4. Métricas de eficacia y seguridad

Una métrica sin **línea base** no dice nada. La línea base es el valor de referencia (por ejemplo, la semana 1 o el proceso manual antes de Nexo).

| Métrica | Qué mide | Por qué importa |
|---|---|---|
| Tasa de resolución autónoma | % de casos de nivel 1 | Eficacia: cuánto trabajo ahorra |
| Tasa de corrección humana | % de respuestas que una persona tuvo que corregir | Calidad: si sube, el agente empeoró |
| Tasa de escalamiento | % de casos de nivel 3 | Carga para el equipo humano |
| Tasa de error técnico | % de fallas del sistema | Confiabilidad |
| Latencia p95 | Tiempo que no supera el 95 % de las respuestas | Experiencia del asociado |

In [ ]:
def metricas(df):
    return pd.Series({
        "consultas": len(df),
        "resolucion_autonoma_%": round(100 * (df.nivel == 1).mean(), 1),
        "correccion_humana_%": round(100 * df.corregido_por_humano.mean(), 1),
        "escalamiento_%": round(100 * (df.nivel == 3).mean(), 1),
        "error_tecnico_%": round(100 * df.error.mean(), 1),
        "latencia_p95_s": round(df.latencia_s.quantile(.95), 2),
    })

tabla = pd.DataFrame({"Semana 1 (línea base)": metricas(sim[sim.semana == 1]),
                      "Semana 2": metricas(sim[sim.semana == 2])})
tabla["Cambio"] = tabla["Semana 2"] - tabla["Semana 1 (línea base)"]
tabla

## 5. Tablero de control

In [ ]:
import matplotlib.pyplot as plt

por_dia = sim.groupby("dia").agg(consultas=("nivel", "size"),
                                 correccion=("corregido_por_humano", "mean"),
                                 escalamiento=("nivel", lambda s: (s == 3).mean()),
                                 p95=("latencia_s", lambda s: s.quantile(.95)))

fig, ejes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle("Tablero de supervisión de Nexo · Cooperativa Raíces (datos simulados)", fontsize=13)

ejes[0, 0].bar(por_dia.index.astype(str), por_dia.consultas, color="#2c90c4")
ejes[0, 0].set_title("Consultas por día")

ejes[0, 1].plot(por_dia.index.astype(str), 100 * por_dia.correccion, marker="o", color="#c0392b")
ejes[0, 1].axhline(10, ls="--", color="gray", label="Umbral de alerta (10 %)")
ejes[0, 1].set_title("Corrección humana (%)"); ejes[0, 1].legend()

ejes[1, 0].plot(por_dia.index.astype(str), 100 * por_dia.escalamiento, marker="o", color="#8e44ad")
ejes[1, 0].set_title("Escalamiento a personas (%)")

ejes[1, 1].plot(por_dia.index.astype(str), por_dia.p95, marker="o", color="#16a085")
ejes[1, 1].set_title("Latencia p95 (segundos)")

for eje in ejes.flat:
    eje.tick_params(axis="x", rotation=60, labelsize=7)
plt.tight_layout()
plt.show()

## 6. Alertas para detectar fallos a tiempo

Un tablero sirve si alguien lo mira. Las **alertas** avisan solas cuando una métrica cruza un **umbral**. Cada alerta debe decir **qué pasó, a quién se avisa y qué hacer**.

In [ ]:
REGLAS_ALERTA = [
    # (nombre, función que calcula el valor del día, umbral, responsable, acción)
    ("Corrección humana alta", lambda d: 100 * d.corregido_por_humano.mean(), 10, "Lucía",
     "Revisar respuestas de precio y actualizar el contexto de Nexo"),
    ("Errores técnicos", lambda d: 100 * d.error.mean(), 5, "Sam",
     "Revisar la conexión con la API y las herramientas"),
    ("Latencia lenta", lambda d: d.latencia_s.quantile(.95), 6, "Sam",
     "Revisar el modelo o reducir el contexto"),
    ("Consultas fuera de horario", lambda d: 100 * d.fuera_de_horario.mean(), 35, "Coordinación",
     "Asegurar que Nexo no prometa recepción de café fuera del horario (Art. 3)"),
]

def revisar_alertas(df):
    alertas = []
    for dia, datos in df.groupby("dia"):
        for nombre, calcular, umbral, responsable, accion in REGLAS_ALERTA:
            valor = calcular(datos)
            if valor > umbral:
                alertas.append({"dia": dia, "alerta": nombre, "valor": round(valor, 1),
                                "umbral": umbral, "avisar_a": responsable, "accion": accion})
    return pd.DataFrame(alertas)

alertas = revisar_alertas(sim)
print("🔔 Alertas generadas:", len(alertas))
alertas

## 7. Delegación progresiva

Un esquema de supervisión no es fijo. La **evidencia del tablero** decide si Nexo gana o pierde autonomía. Por ejemplo: si durante varias semanas Lucía acepta casi todas las propuestas de pago de Nexo, los pagos menores pueden pasar a nivel 1; si sube la corrección humana, vuelve la aprobación previa.

In [ ]:
def recomendar_autonomia(df_semana, umbral_ampliar=3, umbral_reducir=10):
    tasa = 100 * df_semana.corregido_por_humano.mean()
    if tasa <= umbral_ampliar:
        return f"Corrección {tasa:.1f} % ≤ {umbral_ampliar} % → se puede AMPLIAR la autonomía (con seguimiento)."
    if tasa >= umbral_reducir:
        return f"Corrección {tasa:.1f} % ≥ {umbral_reducir} % → REDUCIR la autonomía: volver a aprobación previa."
    return f"Corrección {tasa:.1f} % → MANTENER el esquema actual."

for semana in (1, 2):
    precios = sim[(sim.semana == semana) & (sim.categoria == "precio")]
    print(f"Semana {semana} · consultas de precio: {recomendar_autonomia(precios)}")

## 8. Tu esquema de supervisión humano-IA

El esquema reúne en una sola página todas las piezas: roles, puntos de intervención, escalamiento, métricas, trazabilidad, alertas y delegación. Llénalo para tu agente (aquí está el de Nexo como ejemplo) y llévalo a tu documento final.

In [ ]:
from IPython.display import Markdown, display

ESQUEMA = {
    "Agente(s) y patrón": "Nexo: enrutador + especialistas de Precios, Logística y Pagos",
    "Personas y roles": "Lucía (aprueba pagos ≥ 5 M), Sam (operación técnica), equipo de atención (quejas)",
    "Nivel 1 · Autónomo": "Precio vigente, horarios, requisitos de recogida",
    "Nivel 2 · Con aprobación": "Pagos ≥ 5.000.000 (Art. 6), anticipos de cosecha",
    "Nivel 3 · Escalado": "Quejas (Art. 9), casos no previstos en el reglamento",
    "Métricas y línea base": "Corrección humana (base 5 %), escalamiento, error técnico, latencia p95",
    "Trazabilidad": "Archivo JSONL con id, momento, agente, entrada, decisión, nivel y corrección",
    "Alertas": "Corrección > 10 % → Lucía; errores > 5 % → Sam; latencia p95 > 6 s → Sam",
    "Delegación progresiva": "Corrección ≤ 3 % por 4 semanas → ampliar; ≥ 10 % → volver a aprobación previa",
    "Riesgos éticos": "Datos personales (Art. 8), cifras inventadas, trato desigual por vereda",
}
display(Markdown("| Elemento | Definición |\n|---|---|\n" + "\n".join(f"| {k} | {v} |" for k, v in ESQUEMA.items())))

## 9. Del laboratorio al proyecto final 🎯

Tu **documento del agente** es un cuaderno de Colab como este. Usa la plantilla `Plantilla_Proyecto_Final_Agente.ipynb` y verifica antes de entregar:

- [ ] El cuaderno se ejecuta completo, de arriba hacia abajo, sin errores (*Entorno de ejecución → Reiniciar y ejecutar todas*).
- [ ] La clave de la API está en los **Secretos** de Colab, **no** escrita en el código.
- [ ] Están las nueve secciones del instructivo, con texto (celdas de texto) y evidencias (celdas de código con salida).
- [ ] Hay una tabla con mínimo cinco casos de prueba, incluido un escalamiento y un intento de uso indebido.
- [ ] Guardaste el PDF con todas las salidas visibles y descargaste el `.ipynb`.

**Lecturas de apoyo:** Chang et al. (2024); Mohammadi et al. (2025); Chan et al. (2024); Raji et al. (2020); Breck et al. (2017); Gama et al. (2014); Shneiderman (2020).

## 📄 ¿Cómo guardar este cuaderno en PDF con todas las salidas?

1. *Entorno de ejecución → Reiniciar y ejecutar todas* (o *Ejecutar todas*). Espera a que todas las celdas terminen.
2. Revisa que cada celda muestre su salida y que no haya errores en rojo.
3. *Archivo → Imprimir* y, en *Destino*, elige **Guardar como PDF**.
4. Para el `.ipynb`: *Archivo → Descargar → Descargar .ipynb*.